In [1]:
# This code is a Python script.
# It is intended to be run in a Python 3 environment.

# Compute SOAP descriptors for a set of atomic structures
# Input files are in VASP 'vasprun.xml' format

In [1]:
# Import necessary libraries
from ase.io import read
import numpy as np
from dscribe.descriptors import SOAP
from sklearn.preprocessing import StandardScaler
import glob
import os
from tqdm import tqdm
from datetime import datetime

In [3]:
# Define the path to the directory containing the VASP data files

path_to_data = '/Volumes/LaCie/postdoc_PhLAM/NO_HOPG/data/aimd_nspin_irene_0625_100traj/continue.vaspdata.Ei.0.1.Ts.100.NO.rand.zpe/'

if not os.path.isdir(path_to_data):
    raise NotADirectoryError(f"{path_to_data} is not a valid directory.")
else:
    print(f"Directory {path_to_data} exists and is selected.")

Directory /Volumes/LaCie/postdoc_PhLAM/NO_HOPG/data/aimd_nspin_irene_0625_100traj/continue.vaspdata.Ei.0.1.Ts.100.NO.rand.zpe/ exists and is selected.


In [4]:
# List all vasprun.xml files in the directory

vasp_files = sorted(glob.glob(f"{path_to_data}vasprun-*.xml"))

if not vasp_files:
    raise FileNotFoundError(f"No vasprun.xml files found in {path_to_data}.")
else:
    print(f"Found {len(vasp_files)} vasprun files.")

Found 86 vasprun files.


In [5]:
# Read all structures from the VASP files with ASE
structures = [read(f, index=':') for f in vasp_files]
print(f"Total number of structures read: {sum(len(s) for s in structures)}")

# Get unique atomic species in the structures
species = list(set(atom.symbol for struct_list in structures for atoms in struct_list for atom in atoms))
print(f"Species found in structures: {species}")

Total number of structures read: 113991
Species found in structures: ['C', 'O', 'N']


In [8]:
# Convert the different species into a single one (e.g., 'C')
modify_species = False  # Set to False to keep original species

if modify_species == True:
    print("Modifying all species to a single type.")
    target_species = 'C'
    for struct_list in structures:
        for atoms in struct_list:
            for atom in atoms:
                if atom.symbol != target_species:
                    atom.symbol = target_species

    new_species = list(set(atom.symbol for struct_list in structures for atoms in struct_list for atom in atoms))
    print(f"Species after conversion: {new_species}")
else:
    print("No species modification applied.")
    new_species = species

No species modification applied.


In [ ]:
# define a timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# Define SOAP parameters
soap_params = {
    "species": new_species,
    "periodic": True,
    "r_cut": 6.0,
    "n_max": 4,
    "l_max": 4,
    "sigma": 1.0,
    "compression": {"mode": "mu2"} # Use "mu2" compression to reduce descriptor size
}

if "compression" in soap_params:
    print(f"Using compression mode: {soap_params['compression']['mode']}")

#"mu2" : The SOAP feature vector is generated in an element-agnostic way, 
# so that the size of the feature vector is now independent of the number 
# of elements (see Darby et al. below for details). It is still possible 
# when using this option to construct a feature vector that distinguishes 
# between elements by supplying element-specific weighting under “species_weighting”

#the paper: Darby, J.P., Kermode, J.R. & Csányi, G. Compressing local atomic neighbourhood 
# descriptors. npj Comput Mater 8, 166 (2022). https://doi.org/10.1038/s41524-022-00847-y

# Initialize SOAP descriptor
soap = SOAP(**soap_params)
print("SOAP descriptor initialized with parameters:", soap_params)

# Count total number of structures beforehand
total_structs = sum(len(s) for s in structures)

all_soap_descriptors = []
with tqdm(total=total_structs, desc="Computing SOAP") as pbar:
    for struct_list in structures:
        for atoms in struct_list:
            desc = soap.create(atoms, n_jobs=1)
            all_soap_descriptors.append(desc)
            pbar.update(1)

all_soap_descriptors = np.vstack(all_soap_descriptors)
print(f"Total SOAP descriptors computed: {all_soap_descriptors.shape[0]}")

# --- Filenames
species_str = "-".join(new_species)
npy_file = os.path.join(path_to_data, f"SOAP_{species_str}.npy")

# --- Save npy
np.save(npy_file, all_soap_descriptors)
print(f"SOAP descriptors saved to {npy_file}")

# --- Save metadata
txt_file = os.path.join(
    path_to_data,
    f"SOAP_{species_str}_params_{timestamp}.txt"
)

with open(txt_file, "w") as f:
    f.write("SOAP descriptor computation log\n")
    f.write("---------------------------------\n")
    f.write(f"Timestamp: {timestamp}\n\n")
    f.write(f"path_to_data = '{path_to_data}'\n")
    f.write(f"Number of vasprun.xml files: {len(vasp_files)}\n\n")
    for k, v in soap_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nTotal structures: {total_structs}\n")
    f.write(f"Total descriptors: {all_soap_descriptors.shape[0]}\n")
    f.write(f"Descriptor dimension: {all_soap_descriptors.shape[1]}\n")
    f.write(f"\nData file: {os.path.basename(npy_file)}\n")

print(f"Computation details saved to {txt_file}")


SOAP descriptor initialized with parameters: {'species': ['C', 'O', 'N'], 'periodic': True, 'r_cut': 6.0, 'n_max': 4, 'l_max': 4, 'sigma': 1.0, 'compression': {'mode': 'mu2'}}


Computing SOAP: 100%|██████████| 113991/113991 [06:47<00:00, 279.84it/s]


Total SOAP descriptors computed: 11171118


In [10]:
# SOAP matrix shape
print(f"SOAP descriptor matrix shape: {all_soap_descriptors.shape}")

SOAP descriptor matrix shape: (11171118, 50)


In [ ]:
# Standardize the descriptors
scaler = StandardScaler()
standardized_descriptors = scaler.fit_transform(all_soap_descriptors)
print("Descriptors standardized.")
print(f"Standardized descriptors shape: {standardized_descriptors.shape}")

Descriptors standardized.
Standardized descriptors shape: (11171118, 50)


: 

In [ ]:
X = standardized_descriptors

# Choose: "pca", "umap", or "tsne"
method = "pca"          # change as needed
random_state = 42

# Optional: fast pre-reduction for UMAP/t-SNE on large SOAP
pca_prereduce_dim = None   # set None to skip

embedding = None
model = None

if method.lower() == "pca":
    from sklearn.decomposition import PCA
    n_components = 0.95         # embedding dimension
    model = PCA(n_components=n_components, random_state=random_state)
    embedding = model.fit_transform(X)

elif method.lower() == "umap":
    n_components = 10
    try:
        import umap.umap_ as umap
    except ImportError:
        raise RuntimeError("UMAP not installed. pip install umap-learn")

    X_in = X
    if pca_prereduce_dim:
        X_in = PCA(n_components=min(pca_prereduce_dim, X.shape[1]), random_state=random_state).fit_transform(X)

    model = umap.UMAP(
        n_components=n_components,
        n_neighbors=15,        # tune per dataset size
        min_dist=0.0,
        metric="euclidean",
        random_state=random_state,
        verbose=True
    )
    embedding = model.fit_transform(X_in)

elif method.lower() == "tsne":
    n_components = 10 
    from sklearn.manifold import TSNE
    # t-SNE is O(N^2). Use PCA pre-step by default.
    X_in = X
    if pca_prereduce_dim:
        X_in = PCA(n_components=min(pca_prereduce_dim, X.shape[1]), random_state=random_state).fit_transform(X)

    model = TSNE(
        n_components=n_components,
        perplexity=30,         # 5–50 typical
        n_iter=1000,
        learning_rate="auto",
        init="pca",
        random_state=random_state,
        verbose=1,
        method="barnes_hut" if X_in.shape[0] < 50000 else "exact"
    )
    embedding = model.fit_transform(X_in)

else:
    raise ValueError("method must be 'pca', 'umap', or 'tsne'")

print(f"{method.upper()} embedding shape:", embedding.shape)

/Users/samuel/Desktop/postdoc_PhLAM/codes/data_sampling/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/samuel/Desktop/postdoc_PhLAM/codes/data_sampling/.venv/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(min_dist=0.0, n_components=5, n_jobs=1, random_state=42, verbose=True)
Tue Sep 16 14:31:30 2025 Construct fuzzy simplicial set
Tue Sep 16 14:31:31 2025 Finding Nearest Neighbors
Tue Sep 16 14:31:31 2025 Building RP forest with 64 trees
